> **Daffa was here**

In [1]:
from pathlib import Path
import json
import os
import random

import numpy as np
import pandas as pd

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

currentRoot = Path.cwd().resolve()
projectRoot = currentRoot.parent if currentRoot.name == 'notebooks' else currentRoot
if not (projectRoot / 'notebooks').exists():
    raise RuntimeError('Run this notebook from the project root or notebooks directory.')

dataRoot = projectRoot / 'data'
rawRoot = dataRoot / 'raw'
interimRoot = dataRoot / 'interim'
processedRoot = dataRoot / 'processed'
figureRoot = projectRoot / 'reports' / 'figures'
for outputRoot in (rawRoot, interimRoot, processedRoot, figureRoot):
    outputRoot.mkdir(parents=True, exist_ok=True)

configuredRawRoot = os.environ.get('SKIN_DATA_ROOT')
rawCandidates = [Path(configuredRawRoot).expanduser()] if configuredRawRoot else []
rawCandidates.extend([rawRoot, projectRoot.parent / 'Data'])

def containsDatasetFiles(path: Path) -> bool:
    datasetSuffixes = {'.csv', '.zip', '.jpg', '.jpeg', '.png', '.tif', '.tiff'}
    return path.exists() and any(
        candidate.is_file() and candidate.suffix.lower() in datasetSuffixes
        for candidate in path.rglob('*')
    )

sourceDataRoot = next((path.resolve() for path in rawCandidates if containsDatasetFiles(path)), None)
if sourceDataRoot is None:
    raise FileNotFoundError('No raw dataset found. Populate data/raw or set SKIN_DATA_ROOT.')

def toProjectPath(path: Path) -> str:
    return Path(os.path.relpath(path.resolve(), projectRoot)).as_posix()

def fromProjectPath(pathValue: str) -> Path:
    return (projectRoot / Path(pathValue)).resolve()

print(f'Project root: {projectRoot}')
print(f'Raw dataset root: {sourceDataRoot}')

Project root: D:\sunib\Research\Skin\skin-domain-shift
Raw dataset root: D:\sunib\Research\Skin\Data


In [2]:
import re
import shutil
import zipfile

imageSuffixes = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}

def snakeToCamel(columnName: str) -> str:
    parts = columnName.strip().split('_')
    return parts[0].lower() + ''.join(part[:1].upper() + part[1:] for part in parts[1:])

def safeExtractArchive(archivePath: Path, destinationRoot: Path) -> dict[str, int]:
    destinationRoot.mkdir(parents=True, exist_ok=True)
    extractedCount = 0
    skippedCount = 0
    with zipfile.ZipFile(archivePath) as archive:
        for member in archive.infolist():
            if member.is_dir():
                continue
            memberPath = Path(member.filename.replace('\\', '/'))
            if memberPath.is_absolute() or '..' in memberPath.parts:
                raise ValueError(f'Unsafe archive member in {archivePath.name}: {member.filename}')
            destinationPath = (destinationRoot / memberPath).resolve()
            if destinationRoot.resolve() not in destinationPath.parents:
                raise ValueError(f'Archive path escapes destination: {member.filename}')
            destinationPath.parent.mkdir(parents=True, exist_ok=True)
            if destinationPath.exists() and destinationPath.stat().st_size == member.file_size:
                skippedCount += 1
                continue
            with archive.open(member) as sourceStream, destinationPath.open('wb') as destinationStream:
                shutil.copyfileobj(sourceStream, destinationStream)
            extractedCount += 1
    return {'extractedCount': extractedCount, 'skippedCount': skippedCount}

archivePaths = sorted(sourceDataRoot.rglob('*.zip'))
csvPaths = sorted(sourceDataRoot.rglob('*.csv'))
existingImagePaths = sorted(path for path in sourceDataRoot.rglob('*') if path.is_file() and path.suffix.lower() in imageSuffixes)

clinicalImageRoot = interimRoot / 'clinicalImages'
extractionResults = {}
for archivePath in archivePaths:
    extractionResults[archivePath.name] = safeExtractArchive(archivePath, clinicalImageRoot)

clinicalImagePaths = sorted(path for path in clinicalImageRoot.rglob('*') if path.is_file() and path.suffix.lower() in imageSuffixes)
print(f'Archives: {len(archivePaths)}, CSV files: {len(csvPaths)}, existing images: {len(existingImagePaths):,}')
print(f'Extracted clinical images available: {len(clinicalImagePaths):,}')
extractionResults

Archives: 3, CSV files: 5, existing images: 33,569
Extracted clinical images available: 2,298


{'imgs_part_1.zip': {'extractedCount': 0, 'skippedCount': 911},
 'imgs_part_2.zip': {'extractedCount': 0, 'skippedCount': 659},
 'imgs_part_3.zip': {'extractedCount': 0, 'skippedCount': 728}}

In [3]:
clinicalMetadataCandidates = [path for path in csvPaths if path.name.lower() == 'metadata.csv' and 'test' in path.as_posix().lower()]
if len(clinicalMetadataCandidates) != 1:
    raise RuntimeError(f'Expected one clinical metadata.csv, found {len(clinicalMetadataCandidates)}.')

clinicalMetadataPath = clinicalMetadataCandidates[0]
clinicalMetadataRaw = pd.read_csv(clinicalMetadataPath, dtype={'patient_id': 'string', 'lesion_id': 'string', 'img_id': 'string'})
requiredClinicalColumns = {'img_id', 'patient_id', 'lesion_id', 'diagnostic'}
missingClinicalColumns = requiredClinicalColumns - set(clinicalMetadataRaw.columns)
if missingClinicalColumns:
    raise ValueError(f'Clinical metadata is missing columns: {sorted(missingClinicalColumns)}')

clinicalMetadata = clinicalMetadataRaw.rename(columns={column: snakeToCamel(column) for column in clinicalMetadataRaw.columns})
clinicalPathByName = {path.name.casefold(): path for path in clinicalImagePaths}
clinicalInventory = clinicalMetadata.copy()
clinicalInventory.insert(0, 'imageId', clinicalInventory['imgId'].map(lambda value: Path(str(value)).stem))
clinicalInventory.insert(1, 'imagePath', clinicalInventory['imgId'].map(
    lambda value: toProjectPath(clinicalPathByName[str(value).casefold()]) if str(value).casefold() in clinicalPathByName else pd.NA
))
clinicalInventory.insert(2, 'sourceDataset', 'clinical')
clinicalInventory.insert(3, 'originalLabel', clinicalInventory['diagnostic'].astype('string').str.upper())
clinicalInventory.insert(4, 'harmonizedLabel', pd.NA)
clinicalInventory['patientId'] = clinicalInventory['patientId'].astype('string')
clinicalInventory['lesionId'] = clinicalInventory['lesionId'].astype('string')
clinicalInventory['split'] = pd.NA
clinicalInventory['metadataAvailable'] = True
clinicalInventory['sourceMetadataRow'] = np.arange(2, len(clinicalInventory) + 2)

clinicalMetadataNames = set(clinicalMetadata['imgId'].astype('string').str.casefold())
unmatchedClinicalImages = [path for path in clinicalImagePaths if path.name.casefold() not in clinicalMetadataNames]
missingClinicalImages = int(clinicalInventory['imagePath'].isna().sum())
if clinicalInventory['imageId'].duplicated().any():
    print('Warning: duplicate clinical image IDs were retained for validation.')

clinicalInventoryPath = interimRoot / 'clinicalInventory.csv'
clinicalInventory.to_csv(clinicalInventoryPath, index=False)
print(f'Clinical rows: {len(clinicalInventory):,}; missing images: {missingClinicalImages}; unmatched images: {len(unmatchedClinicalImages)}')
clinicalInventory[['imageId', 'patientId', 'originalLabel', 'imagePath']].head()

Clinical rows: 2,298; missing images: 0; unmatched images: 0


,imageId,patientId,originalLabel,imagePath
0,PAT_1516_1765_530,PAT_1516,NEV,data/interim/clinicalImages/imgs_part_3/PAT_15...
1,PAT_46_881_939,PAT_46,BCC,data/interim/clinicalImages/imgs_part_1/PAT_46...
2,PAT_1545_1867_547,PAT_1545,ACK,data/interim/clinicalImages/imgs_part_3/PAT_15...
3,PAT_1989_4061_934,PAT_1989,ACK,data/interim/clinicalImages/imgs_part_3/PAT_19...
4,PAT_684_1302_588,PAT_684,BCC,data/interim/clinicalImages/imgs_part_2/PAT_68...


In [4]:
isicMetadataPaths = sorted(path for path in csvPaths if 'isic' in path.name.lower() and 'metadata' in path.name.lower())
groundTruthPatterns = ('groundtruth', 'ground_truth', 'training_truth', 'training_labels')
isicGroundTruthPaths = sorted(
    path for path in csvPaths
    if 'isic' in path.name.lower() and any(pattern in path.name.lower() for pattern in groundTruthPatterns)
)

isicFrames = []
for metadataPath in isicMetadataPaths:
    metadataFrame = pd.read_csv(metadataPath)
    if 'image' not in metadataFrame.columns:
        continue
    partitionName = 'officialTest' if 'test' in metadataPath.name.lower() else 'officialTrain'
    candidateRoots = sorted({path.parent for path in existingImagePaths if 'isic' in path.name.lower()})
    imageById = {
        path.stem.casefold(): path
        for candidateRoot in candidateRoots
        for path in candidateRoot.glob('*')
        if path.is_file() and path.suffix.lower() in imageSuffixes
    }
    normalizedFrame = metadataFrame.rename(columns={column: snakeToCamel(column) for column in metadataFrame.columns})
    normalizedFrame.insert(0, 'imageId', normalizedFrame['image'].astype('string'))
    normalizedFrame.insert(1, 'imagePath', normalizedFrame['imageId'].map(
        lambda value: toProjectPath(imageById[str(value).casefold()]) if str(value).casefold() in imageById else pd.NA
    ))
    normalizedFrame.insert(2, 'sourceDataset', 'isic2019')
    normalizedFrame.insert(3, 'originalLabel', pd.NA)
    normalizedFrame.insert(4, 'harmonizedLabel', pd.NA)
    normalizedFrame.insert(5, 'patientId', pd.NA)
    if 'lesionId' not in normalizedFrame.columns:
        normalizedFrame.insert(6, 'lesionId', pd.NA)
    normalizedFrame['split'] = partitionName
    normalizedFrame['officialPartition'] = partitionName
    normalizedFrame['metadataAvailable'] = True
    normalizedFrame['sourceMetadataFile'] = metadataPath.name
    isicFrames.append(normalizedFrame)

isicInventory = pd.concat(isicFrames, ignore_index=True) if isicFrames else pd.DataFrame()

labelColumns = ['MEL', 'NV', 'BCC', 'AK', 'BKL', 'DF', 'VASC', 'SCC', 'UNK']
groundTruthFrames = []
for groundTruthPath in isicGroundTruthPaths:
    groundTruthFrame = pd.read_csv(groundTruthPath)
    presentLabelColumns = [column for column in labelColumns if column in groundTruthFrame.columns]
    if 'image' not in groundTruthFrame.columns or not presentLabelColumns:
        continue
    rowTotals = groundTruthFrame[presentLabelColumns].fillna(0).sum(axis=1)
    if not rowTotals.eq(1).all():
        raise ValueError(f'Ground truth is not one-hot encoded: {groundTruthPath.name}')
    groundTruthFrame['originalLabel'] = groundTruthFrame[presentLabelColumns].idxmax(axis=1)
    groundTruthFrame['sourceGroundTruthFile'] = groundTruthPath.name
    groundTruthFrames.append(groundTruthFrame[['image', 'originalLabel', 'sourceGroundTruthFile']])
if groundTruthFrames:
    combinedGroundTruth = pd.concat(groundTruthFrames, ignore_index=True)
    if combinedGroundTruth['image'].duplicated().any():
        raise ValueError('ISIC ground-truth files contain duplicate image IDs.')
    labelByImage = combinedGroundTruth.set_index('image')['originalLabel']
    groundTruthFileByImage = combinedGroundTruth.set_index('image')['sourceGroundTruthFile']
    isicInventory['originalLabel'] = isicInventory['imageId'].map(labelByImage)
    isicInventory['sourceGroundTruthFile'] = isicInventory['imageId'].map(groundTruthFileByImage)

isicInventoryPath = interimRoot / 'isicInventory.csv'
isicInventory.to_csv(isicInventoryPath, index=False)
isicMissingImages = int(isicInventory['imagePath'].isna().sum()) if len(isicInventory) else 0
isicLabelledCount = int(isicInventory['originalLabel'].notna().sum()) if len(isicInventory) else 0
print(f'ISIC rows: {len(isicInventory):,}; missing images: {isicMissingImages}; labelled rows: {isicLabelledCount:,}')
if isicLabelledCount != len(isicInventory):
    print('MISSING PREREQUISITE: official ground truth does not cover every ISIC image.')
isicInventory[['imageId', 'officialPartition', 'originalLabel', 'imagePath']].head()

ISIC rows: 33,569; missing images: 0; labelled rows: 33,569


,imageId,officialPartition,originalLabel,imagePath
0,ISIC_0000000,officialTrain,NV,../Data/Train/ISIC_2019_Training_Input/ISIC_00...
1,ISIC_0000001,officialTrain,NV,../Data/Train/ISIC_2019_Training_Input/ISIC_00...
2,ISIC_0000002,officialTrain,MEL,../Data/Train/ISIC_2019_Training_Input/ISIC_00...
3,ISIC_0000003,officialTrain,NV,../Data/Train/ISIC_2019_Training_Input/ISIC_00...
4,ISIC_0000004,officialTrain,MEL,../Data/Train/ISIC_2019_Training_Input/ISIC_00...


In [5]:
import tempfile

with tempfile.TemporaryDirectory() as temporaryName:
    temporaryRoot = Path(temporaryName)
    unsafeArchive = temporaryRoot / 'unsafe.zip'
    with zipfile.ZipFile(unsafeArchive, 'w') as archive:
        archive.writestr('../escape.txt', 'unsafe')
    try:
        safeExtractArchive(unsafeArchive, temporaryRoot / 'output')
        raise AssertionError('Unsafe archive member was not rejected.')
    except ValueError:
        pass

discoveryReport = {
    'sourceDataRoot': toProjectPath(sourceDataRoot),
    'archiveCount': len(archivePaths),
    'csvCount': len(csvPaths),
    'clinical': {
        'metadataRows': len(clinicalInventory),
        'imageCount': len(clinicalImagePaths),
        'missingImages': missingClinicalImages,
        'unmatchedImages': len(unmatchedClinicalImages),
        'imageIdColumn': 'img_id',
        'patientIdColumn': 'patient_id',
    },
    'isic': {
        'metadataRows': len(isicInventory),
        'imageCount': int(isicInventory['imagePath'].notna().sum()),
        'missingImages': isicMissingImages,
        'groundTruthFiles': [toProjectPath(path) for path in isicGroundTruthPaths],
        'labelledRows': isicLabelledCount,
        'missingPrerequisites': [] if isicLabelledCount == len(isicInventory) else ['Official ISIC ground truth covering every inventoried image'],
    },
    'outputs': [toProjectPath(clinicalInventoryPath), toProjectPath(isicInventoryPath)],
}
discoveryReportPath = interimRoot / 'datasetDiscoveryReport.json'
discoveryReportPath.write_text(json.dumps(discoveryReport, indent=2), encoding='utf-8')
print(json.dumps(discoveryReport, indent=2))

{
  "sourceDataRoot": "../Data",
  "archiveCount": 3,
  "csvCount": 5,
  "clinical": {
    "metadataRows": 2298,
    "imageCount": 2298,
    "missingImages": 0,
    "unmatchedImages": 0,
    "imageIdColumn": "img_id",
    "patientIdColumn": "patient_id"
  },
  "isic": {
    "metadataRows": 33569,
    "imageCount": 33569,
    "missingImages": 0,
    "groundTruthFiles": [
      "../Data/Train/ISIC_2019_Training_GroundTruth.csv",
      "../Data/Validation/ISIC_2019_Test_GroundTruth.csv"
    ],
    "labelledRows": 33569,
    "missingPrerequisites": []
  },
  "outputs": [
    "data/interim/clinicalInventory.csv",
    "data/interim/isicInventory.csv"
  ]
}
